# Govern the machine-shop graph

Validate instances with SHACL shapes, shop-floor Python rules, `GraphValidator`, and `OntologyQualityGate`. The violation seed (machine without a work center, negative tool life) must fail the gate.

In [ ]:
import sys
from pathlib import Path

HERE = Path.cwd()
if (HERE / "pipeline").exists():
    ROOT = HERE
else:
    ROOT = HERE / "cookbook" / "use_cases" / "manufacturing"
sys.path.insert(0, str(ROOT.parent))

from manufacturing.pipeline.govern import govern
from manufacturing.pipeline.ingest_and_link import ingest_and_link
from manufacturing.sample.build_sample_db import build_sample_db

In [ ]:
clean_db = ROOT / "sample" / "machine_shop.sqlite"
build_sample_db(clean_db, include_violations=False)
clean_graph = ingest_and_link(clean_db, include_violations=False, build_if_missing=False)
clean_report = govern(clean_graph)
print("clean passed", clean_report["passed"])
print("shop_rules", clean_report["shop_rules"]["passed"])
print("shacl", clean_report["shacl"].get("conforms"), "available=", clean_report["shacl"].get("available"))
print("quality_gate", clean_report["quality_gate"].get("passed"))

In [ ]:
dirty_db = ROOT / "sample" / "machine_shop_violations.sqlite"
build_sample_db(dirty_db, include_violations=True)
dirty_graph = ingest_and_link(dirty_db, include_violations=True, build_if_missing=False)
dirty_report = govern(dirty_graph)
print("dirty passed", dirty_report["passed"])
for issue in dirty_report["shop_rules"]["issues"]:
    print(issue["code"], issue["message"], issue.get("element_id"))